# Fig 3: Cooling technologies vs. rack power density (paper version)

Each source's stated range for each cooling technology, from `Data/generated/cooling_regimes.csv`. This is a forest-plot style figure: one thin bar per source, grouped into technology rows. It shows where sources overlap and where they disagree, instead of drawing hard thresholds (Section 2.3: *overlapping operating regimes, not universal thresholds*).

How each kind of value is drawn:
- **Stated range** (low and high): solid bar.
- **"Up to X"** (no low stated): short dotted lead-in ending in a cap at X. The source states a ceiling, not a range, so we don't draw down to the axis start.
- **"X and above"** (no high stated): solid bar from X, with an arrow.
- **Single design point** (`kW_typical` only): diamond.

Sources that don't name a sub-type (e.g. "air cooling" generally) get their own row instead of being assigned one.
Caveats for the caption: Vertiv 2024 chart values were read off a bar chart (approximate). Immersion research values are per tank, not per rack. Vendor figures are product claims.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
from pathlib import Path
import paper_style as ps

ps.apply()
DATA = Path('../Data')
OUT = Path('../figures'); OUT.mkdir(exist_ok=True)

# Ordinal ramp by how close the liquid gets to the chip: air -> liquid-assisted air -> liquid
C_AIR, C_RDHX, C_LIQ = '#8a8984', '#7fcfb0', '#1baf7a'

## Data: map technologies to display rows

In [ ]:
reg = pd.read_csv(DATA / 'generated/cooling_regimes.csv')

# display row, row colour, and a suffix added to the source label to keep sub-type information
ROW = {
    'Air cooling, traditional (perimeter CRAC/CRAH, raised floor, no containment)': ('Air: room-level', C_AIR, ''),
    'Air cooling with hot/cold aisle containment': ('Air: containment / in-row', C_AIR, ', containment'),
    'In-row / close-coupled air cooling': ('Air: containment / in-row', C_AIR, ', in-row/in-rack'),
    'Air cooling (general, type unspecified)': ('Air: type not stated', C_AIR, ''),
    'Rear-door heat exchanger': ('Rear-door heat exchanger', C_RDHX, ''),
    'Direct-to-chip liquid cooling, single-phase (cold plates)': ('Direct-to-chip liquid', C_LIQ, ', 1-phase'),
    'Direct-to-chip liquid cooling, two-phase': ('Direct-to-chip liquid', C_LIQ, ', 2-phase'),
    'Direct-to-chip liquid cooling (phase unspecified)': ('Direct-to-chip liquid', C_LIQ, ''),
    'Immersion cooling, single-phase': ('Immersion', C_LIQ, ', 1-phase'),
    'Immersion cooling, two-phase': ('Immersion', C_LIQ, ', 2-phase'),
    'Immersion cooling (phase unspecified)': ('Immersion', C_LIQ, ''),
    'Liquid cooling (type unspecified)': ('Liquid: type not stated', C_LIQ, ''),
}
ROW_ORDER = ['Air: room-level', 'Air: containment / in-row', 'Air: type not stated', 'Rear-door heat exchanger',
             'Direct-to-chip liquid', 'Immersion', 'Liquid: type not stated']

SHORT_SOURCE = {  # source -> short label
    'Servnet tracker (2026)': 'Servnet 2026', 'SERES rack densities (2026)': 'SERES 2026',
    'Uptime Cooling Systems Survey (2024)': 'Uptime survey 2024', 'Schneider Electric blog (2021)': 'Schneider 2021',
    'Vertiv containment article (2025)': 'Vertiv 2025', 'Chakraborty et al., UC Davis WCEC (2026)': 'UC Davis 2026',
    'Vertiv WP (2024), Fig.': 'Vertiv 2024 (chart)', 'Vertiv WP (2024)': 'Vertiv 2024',
    'Schneider Electric blog (2011)': 'Schneider 2011', 'Uptime Institute journal (2020)': 'Uptime 2020',
    'ENERGY STAR in-row/in-rack page (n.d.)': 'ENERGY STAR', 'Accelsius press release (2025)': 'Accelsius 2025',
    'GRC blog (2020)': 'GRC 2020', 'Shah & Tuma, Electronics Cooling (2024)': 'Shah & Tuma 2024',
}

def rdhx_type(note):
    n = str(note).lower()
    return ', passive' if 'type: passive' in n else ', active' if 'type: active' in n else ''

def per_tank(note):
    return ', per tank' if 'per tank' in str(note).lower() else ''

unmapped = set(reg['technology']) - set(ROW)
assert not unmapped, unmapped
reg['row'] = reg['technology'].map(lambda t: ROW[t][0])
reg['color'] = reg['technology'].map(lambda t: ROW[t][1])
reg['label'] = (reg['source'].map(SHORT_SOURCE).fillna(reg['source']) + reg['technology'].map(lambda t: ROW[t][2])
                + reg['notes'].map(rdhx_type).where(reg['row'] == 'Rear-door heat exchanger', '')
                + reg['notes'].map(per_tank))
reg['row_rank'] = reg['row'].map(ROW_ORDER.index)
reg = reg.sort_values(['row_rank', 'kW_high', 'kW_low'], na_position='first').reset_index(drop=True)
reg[['row', 'label', 'kW_low', 'kW_high', 'kW_typical', 'source_type']]

## Figure

In [ ]:
X_MIN, X_MAX = 1, 600
AIR_LIMIT = (20, 45)  # where sources place the practical limit of air cooling (Uptime 2020/2024, Vertiv 2025, Schneider 2021, Servnet, UC Davis)
LINE_STEP, ROW_GAP = 1.0, 1.0

# y positions: one line per source, rows separated by a gap, first row at the top
ys, row_spans, y = [], {}, 0
for row in ROW_ORDER:
    idx = reg.index[reg['row'] == row]
    start = y
    for _ in idx:
        ys.append(y); y += LINE_STEP
    row_spans[row] = (start, y - LINE_STEP)
    y += ROW_GAP
reg['y'] = ys

fig, ax = plt.subplots(figsize=(ps.TEXT_W, 3.7))
ax.axvspan(*AIR_LIMIT, color=ps.GRID, alpha=0.6, lw=0, zorder=0)
ax.text(np.sqrt(AIR_LIMIT[0] * AIR_LIMIT[1]), -1.3, 'Stated limits of\nair cooling', ha='center', va='bottom',
        fontsize=6, color=ps.TEXT_2)

for _, r in reg.iterrows():
    lo, hi, typ, c, yy = r['kW_low'], r['kW_high'], r['kW_typical'], r['color'], r['y']
    if pd.notna(lo) and pd.notna(hi):
        ax.plot([max(lo, X_MIN), hi], [yy, yy], color=c, lw=2.2, solid_capstyle='butt', zorder=3)
        end = hi
    elif pd.notna(hi):   # 'up to X': a ceiling, so only a short lead-in to the cap
        ax.plot([hi / 2.5, hi], [yy, yy], color=c, lw=1.2, ls=(0, (1, 1.2)), zorder=3)
        ax.plot([hi, hi], [yy - 0.3, yy + 0.3], color=c, lw=1.2, zorder=3)
        end = hi
    elif pd.notna(lo):   # 'X and above'
        ax.annotate('', xy=(lo * 2.2, yy), xytext=(lo, yy), zorder=3,
                    arrowprops=dict(arrowstyle='-|>', color=c, lw=2.2, mutation_scale=6, shrinkA=0, shrinkB=0))
        end = lo * 2.2
    else:
        end = None
    if pd.notna(typ):
        ax.scatter(typ, yy, marker='D', s=16, color=c, edgecolor='white', linewidth=0.6, zorder=4)
        end = max(end or 0, typ)
    ax.text(end * 1.08, yy, r['label'], va='center', ha='left', fontsize=5.5, color=ps.TEXT_2)

ax.set_xscale('log')
ax.xaxis.set_major_locator(FixedLocator([1, 2, 5, 10, 20, 50, 100, 200, 500]))
ax.xaxis.set_minor_locator(NullLocator())
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.set_xlim(X_MIN, X_MAX)
ax.set_xlabel('Rack power density (kW per rack, log scale)')
ax.set_ylim(y - ROW_GAP + 0.5, -1.5)
ax.set_yticks([(a + b) / 2 for a, b in row_spans.values()])
ax.set_yticklabels(list(row_spans), fontsize=6.5, color=ps.TEXT)
ax.grid(axis='y', visible=False)
ax.grid(axis='x', color=ps.GRID, lw=0.5)
for row, (a, b) in list(row_spans.items())[1:]:   # thin separators between technology rows
    ax.axhline(a - (LINE_STEP + ROW_GAP) / 2, color=ps.GRID, lw=0.6, zorder=1)

handles = [Line2D([], [], color=ps.TEXT_2, lw=2.2, label='Stated range'),
           Line2D([], [], color=ps.TEXT_2, lw=1.2, ls=(0, (1, 1.2)), label='"Up to" (ceiling only)'),
           Line2D([], [], color=ps.TEXT_2, lw=0, marker=r'$\rightarrow$', ms=7, label='"And above" (floor only)'),
           Line2D([], [], color=ps.TEXT_2, lw=0, marker='D', ms=3.5, label='Single design point')]
ax.legend(handles=handles, loc='upper right', frameon=False, fontsize=6, handlelength=2.2, borderaxespad=0.1)

fig.savefig(OUT / 'fig3_cooling_regimes.pdf')
fig.savefig(OUT / 'fig3_cooling_regimes.png', dpi=300)
plt.show()

## Source list for the caption / appendix

In [ ]:
reg[['label', 'source', 'source_type', 'year', 'source_url']].drop_duplicates('source').sort_values('source')